# UDCF (LBCF, Ai et al. 2022, WWW'22) aplicado a base Hillstrom - versao ORDINAL

Codigo C++ original dos autores, sem reimplementacao.

Configuracao desta versao (teste de isolamento do efeito da codificacao categorica):
- **Hiperparametros**: defaults dos autores, sem nenhum ajuste (`min_node_size=50`, `alpha=0.05`, `imbalance_penalty=0.01`, `mtry=3`, `num_trees=300`) - igual a versao one-hot
- **Amostra/predicao**: 100% da base, `predict_oob` - igual a versao one-hot
- **Codificacao de `zip_code`/`channel`**: ORDINAL (mesmo mapeamento numerico do script original do usuario: Urban=0/Surburban=1/Rural=2, Phone=0/Web=1/Multichannel=2) -> 7 features no total

A UNICA diferenca entre este notebook e `UDCF_Hillstrom_Colab_OneHot.ipynb` e a codificacao categorica na Secao 3. Tudo o mais (hiperparametros, amostra, predicao, algoritmo C++) e identico. O objetivo e isolar se foi a codificacao (one-hot vs ordinal) que explica a diferenca de resultados observada.

## 1. Clonar o repositorio dos autores e instalar ferramentas de build

In [ ]:
!git clone -q https://github.com/www2022paper/WWW-2022-PAPER-SUPPLEMENTARY-MATERIALS.git
!apt-get -qq update && apt-get -qq install -y cmake g++

## 2. Extrair o codigo C++ do UDCF (vem zipado dentro do repositorio)

In [ ]:
import zipfile

BASE = "WWW-2022-PAPER-SUPPLEMENTARY-MATERIALS/Code/Model/LBCF"
with zipfile.ZipFile(f"{BASE}/LBCF_RCT.zip") as z:
    z.extractall(BASE)

## 3. Carregar a base Hillstrom e montar o arquivo de entrada (codificacao ORDINAL)

- Outcome (Y): `conversion`
- Tratamento multi-nivel (K=2): `No E-Mail` = controle, `Mens E-Mail` = T1, `Womens E-Mail` = T2
- Features: `recency, history, mens, womens, newbie` + `zip_code_num` (1 coluna, 0/1/2) + `channel_num` (1 coluna, 0/1/2) = 7 features

Mapeamento identico ao script original do usuario: `zip_code`: Urban=0, Surburban=1, Rural=2 | `channel`: Phone=0, Web=1, Multichannel=2.

In [ ]:
import pandas as pd

HILLSTROM_URL = (
    "http://www.minethatdata.com/"
    "Kevin_Hillstrom_MineThatData_E-MailAnalytics_DataMiningChallenge_2008.03.20.csv"
)
df = pd.read_csv(HILLSTROM_URL)
print("Base Hillstrom original:", df.shape)

feature_cols = ["recency", "history", "mens", "womens", "newbie"]
X = df[feature_cols].astype(float).copy()

X["zip_code_num"] = df["zip_code"].map({"Urban": 0, "Surburban": 1, "Rural": 2}).astype(float)
X["channel_num"] = df["channel"].map({"Phone": 0, "Web": 1, "Multichannel": 2}).astype(float)

Y = df["conversion"].astype(float)
code = df["segment"].map({"No E-Mail": 0, "Mens E-Mail": 1, "Womens E-Mail": 2})
T1 = (code == 1).astype(float)
T2 = (code == 2).astype(float)

design = pd.concat([X, Y.rename("Y"), T1.rename("T1"), T2.rename("T2")], axis=1)

n_features = X.shape[1]
outcome_index = n_features
treatment_index = [n_features + 1, n_features + 2]

data_path = f"{BASE}/UDCF_RCT/core/hillstrom_udcf_input.txt"
design.to_csv(data_path, sep=" ", header=False, index=False)

print("numero de features:", n_features)
print("outcome_index:", outcome_index, "| treatment_index:", treatment_index)
print("linhas x colunas do arquivo de entrada:", design.shape)

## 4. Gerar o `main.cpp` adaptado

So os indices de coluna e os caminhos de arquivo mudam (calculados automaticamente a partir do numero de features acima). Hiperparametros = defaults dos autores (`ForestTestUtilities::default_options`, sem edicao). Predicao = `predict_oob` (100% da amostra, honesta).

In [ ]:
main_cpp = f'''#include <iostream>
#include <string>
#include <unistd.h>

#include "tree/Tree.h"
#include "prediction/DefaultPredictionStrategy.h"
#include "commons/utility.h"
#include "forest/ForestPredictor.h"
#include "forest/ForestTrainer.h"
#include "utilities/FileTestUtilities.h"
#include "utilities/ForestTestUtilities.h"
#include "forest/ForestTrainers.h"
#include "forest/ForestPredictors.h"
#include "analysis/SplitFrequencyComputer.h"
using namespace grf;

void update_predictions_file(const std::string& file_name,
                             const std::vector<Prediction>& predictions) {{
  std::vector<std::vector<double>> values;
  values.reserve(predictions.size());
  for (const auto& prediction : predictions) {{
    values.push_back(prediction.get_predictions());
  }}
  FileTestUtilities::write_csv_file(file_name, values);
  std::cout << "success! predictions dump to " << file_name << std::endl;
}}

int main()
{{
    auto data_vec = load_data("../hillstrom_udcf_input.txt");
    Data data(data_vec);
    data.set_outcome_index({outcome_index});
    data.set_treatment_index({{{treatment_index[0]}, {treatment_index[1]}}});

    size_t num_treatments = 2;

    ForestTrainer trainer = udcf_trainer(num_treatments, 1, true);
    ForestOptions options = ForestTestUtilities::default_options(true, 1);
    Forest forest = trainer.train(data, options);

    std::cout << "FOREST_INFO num_trees=" << forest.get_trees().size()
               << " num_variables=" << forest.get_num_variables() << std::endl;
    const auto& first_tree = forest.get_trees()[0];
    size_t root = first_tree->get_root_node();
    size_t total_nodes = first_tree->get_child_nodes()[0].size();
    bool root_is_leaf = first_tree->is_leaf(root);
    std::cout << "TREE0_INFO total_nodes=" << total_nodes
               << " root=" << root
               << " root_is_leaf=" << (root_is_leaf ? "true" : "false") << std::endl;
    size_t non_leaf_count = 0;
    for (size_t n = 0; n < total_nodes; n++) {{
      if (!first_tree->is_leaf(n)) {{
        non_leaf_count++;
      }}
    }}
    std::cout << "TREE0_INFO non_leaf_node_count=" << non_leaf_count << std::endl;

    SplitFrequencyComputer freq_computer;
    std::vector<std::vector<size_t>> freq = freq_computer.compute(forest, 30);
    std::vector<size_t> total_per_var(forest.get_num_variables(), 0);
    for (const auto& depth_counts : freq) {{
      for (size_t v = 0; v < depth_counts.size(); v++) {{
        total_per_var[v] += depth_counts[v];
      }}
    }}
    std::cout << "SPLIT_FREQ ";
    for (size_t v = 0; v < total_per_var.size(); v++) {{
      std::cout << v << ":" << total_per_var[v] << " ";
    }}
    std::cout << std::endl;

    ForestPredictor predictor = udcf_predictor(1, num_treatments, 1);

    std::vector<Prediction> predictions = predictor.predict_oob(forest, data, false);
    update_predictions_file("../hillstrom_udcf_predictions.txt", predictions);

    return 0;
}}
'''

with open(f"{BASE}/UDCF_RCT/core/main.cpp", "w") as f:
    f.write(main_cpp)

print("main.cpp gerado com sucesso.")

## 5. Compilar (cmake + make, igual ao README original do repositorio)

In [ ]:
%%bash
cd WWW-2022-PAPER-SUPPLEMENTARY-MATERIALS/Code/Model/LBCF/UDCF_RCT/core
rm -rf build
mkdir build
cd build
cmake .. -DCMAKE_BUILD_TYPE=Release
make -j4

## 6. Rodar o UDCF treinado (o binario original dos autores)

In [ ]:
%%bash
cd WWW-2022-PAPER-SUPPLEMENTARY-MATERIALS/Code/Model/LBCF/UDCF_RCT/core/build
./UDCF_RCT

## 7. Ler o CATE estimado e resumir os resultados

In [ ]:
preds = pd.read_csv(
    f"{BASE}/UDCF_RCT/core/hillstrom_udcf_predictions.txt",
    header=None, sep=r",\s*", engine="python",
)
preds.columns = ["cate_mens_email", "cate_womens_email"]

out = pd.concat([df.reset_index(drop=True), preds], axis=1)
out.to_csv("hillstrom_udcf_cate_ordinal.csv", index=False)

print("=== Resumo do CATE estimado (UDCF, codificacao ORDINAL) ===")
for nome, col in [("Mens E-Mail", "cate_mens_email"), ("Womens E-Mail", "cate_womens_email")]:
    c = out[col]
    print(f"\n{nome}")
    print(f"  media (CATE medio / ATE aproximado): {c.mean():.4f}")
    print(f"  desvio padrao entre usuarios:         {c.std():.4f}")
    print(f"  minimo / maximo:                      {c.min():.4f} / {c.max():.4f}")

naive_t1 = Y[T1 == 1].mean() - Y[(T1 == 0) & (T2 == 0)].mean()
naive_t2 = Y[T2 == 1].mean() - Y[(T1 == 0) & (T2 == 0)].mean()
print("\n=== Diferenca simples de medias (ATE naive, para conferencia) ===")
print(f"  Mens E-Mail vs controle:   {naive_t1:.4f}")
print(f"  Womens E-Mail vs controle: {naive_t2:.4f}")

print("\nArquivo salvo: hillstrom_udcf_cate_ordinal.csv (inclui CATE por usuario)")